# Task 1 - Web Scraping
**CodeAlpha Data Analytics Internship**

Goal: collect a real dataset from the web using Python.

* Website: `http://books.toscrape.com/` (a site built specifically for scraping practice)
* Library: `requests` + `BeautifulSoup`
* Fields extracted: title, price, rating, availability, stock count, category
* Output: `dataset/books.csv`


In [1]:
import os, re, time, csv
import requests
from bs4 import BeautifulSoup

BASE_URL = "http://books.toscrape.com/"
OUTPUT_FILE = "dataset/books.csv"
os.makedirs("dataset", exist_ok=True)

RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def get_soup(url):
    headers = {"User-Agent": "Mozilla/5.0 (CodeAlpha Internship Scraper)"}
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()
    r.encoding = "utf-8"
    return BeautifulSoup(r.text, "html.parser")

def scrape_category(url, name):
    rows, page = [], 1
    while url:
        soup = get_soup(url)
        for a in soup.select("article.product_pod"):
            avail = a.select_one("p.instock.availability").text.strip()
            nums = re.findall(r"\d+", avail)
            rating = next((RATING_MAP[c] for c in a.select_one("p.star-rating")["class"] if c in RATING_MAP), None)
            rows.append({
                "title": a.h3.a["title"],
                "price": float(a.select_one("p.price_color").text.replace("\u00a3", "")),
                "rating": rating,
                "in_stock": "In stock" in avail,
                "stock_count": int(nums[0]) if nums else 0,
                "category": name,
            })
        nxt = soup.select_one("li.next > a")
        url = url.rsplit("/", 1)[0] + "/" + nxt["href"] if nxt else None
        page += 1
        time.sleep(0.1)
    return rows

soup = get_soup(BASE_URL)
categories = [(l.text.strip(), BASE_URL + "catalogue/" + l["href"].replace("catalogue/", "").replace("index.html", ""))
              for l in soup.select("div.side_categories ul li ul li a")]
print(f"Found {len(categories)} categories")

all_rows = []
for name, url in categories:
    all_rows.extend(scrape_category(url, name))

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["title", "price", "rating", "in_stock", "stock_count", "category"])
    w.writeheader()
    w.writerows(all_rows)

print(f"Scraped {len(all_rows)} books -> {OUTPUT_FILE}")

Found 50 categories


Scraped 1000 books -> dataset/books.csv


### Verify the scraped dataset

In [2]:
import pandas as pd
df = pd.read_csv("dataset/books.csv")
print("Shape:", df.shape)
df.head()

Shape: (1000, 6)


,title,price,rating,in_stock,stock_count,category
0,It's Only the Himalayas,45.17,2,True,0,Travel
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,49.43,4,True,0,Travel
2,See America: A Celebration of Our National Par...,48.87,3,True,0,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,2,True,0,Travel
4,Under the Tuscan Sun,37.33,3,True,0,Travel
